In [27]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


from sklearn.metrics import accuracy_score, f1_score, classification_report

In [4]:
df = pd.read_csv("../data/CustomerInteractionData.csv")

X = df["CustomerInteractionRawText"].fillna("")
y = df["AgentAssignedTopic"]

print("Total de conversaciones originales:", len(df))

Total de conversaciones originales: 103


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Datos de entrenamiento:", len(X_train))
print("Datos de prueba:", len(X_test))

Datos de entrenamiento: 82
Datos de prueba: 21


In [6]:
modelo_original = joblib.load(
    "modelo_clasificacion_conversaciones.joblib"
)

predicciones_originales = modelo_original.predict(X_test)

casos_revision = pd.DataFrame({
    "Conversación": X_test.values,
    "Etiqueta_real": y_test.values,
    "Predicción_original": predicciones_originales
})

casos_revision["Correcto"] = (
    casos_revision["Etiqueta_real"] ==
    casos_revision["Predicción_original"]
)

casos_revision.head()

,Conversación,Etiqueta_real,Predicción_original,Correcto
0,All India plans for Data usage. Don't want inc...,Roaming Plans,Data Usage,False
1,Added a new line. Offered the customer new plans.,Add New Line,Add New Line,True
2,Want to know about offers on recent bill as he...,High Bill,High Bill,True
3,Gregory cancelled his services temporarily.,Cancel Service,Cancel Service,True
4,#call?,Others,Others,True


In [7]:
casos_corregidos = casos_revision[
    casos_revision["Correcto"] == False
].copy()

print("Casos corregidos por revisión humana:")
print(len(casos_corregidos))

casos_corregidos[
    [
        "Conversación",
        "Etiqueta_real",
        "Predicción_original"
    ]
]

Casos corregidos por revisión humana:
7


,Conversación,Etiqueta_real,Predicción_original
0,All India plans for Data usage. Don't want inc...,Roaming Plans,Data Usage
6,Customer terminated the offers permanently.,Deactivate,Change Plans
7,935468R911 called to report abrupt disconectio...,Poor Connectivity,Others
8,Wants to access News channels through phone.,Others,Cancel Service
17,John Meyer want to cancel the current connection.,Cancel Service,Poor Connectivity
18,Decativated all plans due to change of user.,Deactivate,Change Plans
19,Customer wants to block calls from a particula...,Others,Deactivate


In [8]:
X_entrenamiento, X_validacion, y_entrenamiento, y_validacion = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Datos para entrenamiento:", len(X_entrenamiento))
print("Datos para validación:", len(X_validacion))
print("Casos corregidos disponibles:", len(casos_corregidos))

Datos para entrenamiento: 65
Datos para validación: 17
Casos corregidos disponibles: 7


In [9]:
X_corregido = casos_corregidos["Conversación"]

y_corregido = casos_corregidos["Etiqueta_real"]

print("Conversaciones corregidas:", len(X_corregido))

Conversaciones corregidas: 7


In [10]:
X_reentrenamiento = pd.concat(
    [
        X_entrenamiento,
        X_corregido
    ],
    ignore_index=True
)

y_reentrenamiento = pd.concat(
    [
        y_entrenamiento,
        y_corregido
    ],
    ignore_index=True
)

print("Total datos para reentrenamiento:", len(X_reentrenamiento))

Total datos para reentrenamiento: 72


In [11]:
modelo_reentrenado = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_reentrenado.fit(
    X_reentrenamiento,
    y_reentrenamiento
)

print("✅ Modelo reentrenado correctamente.")

✅ Modelo reentrenado correctamente.


In [12]:
predicciones_reentrenadas = modelo_reentrenado.predict(
    X_validacion
)

accuracy_reentrenado = accuracy_score(
    y_validacion,
    predicciones_reentrenadas
)

macro_f1_reentrenado = f1_score(
    y_validacion,
    predicciones_reentrenadas,
    average="macro",
    zero_division=0
)

print(f"Accuracy modelo reentrenado: {accuracy_reentrenado:.2%}")
print(f"Macro F1 modelo reentrenado: {macro_f1_reentrenado:.4f}")

Accuracy modelo reentrenado: 82.35%
Macro F1 modelo reentrenado: 0.7133


In [13]:
predicciones_base = modelo_original.predict(X_validacion)

accuracy_base = accuracy_score(
    y_validacion,
    predicciones_base
)

macro_f1_base = f1_score(
    y_validacion,
    predicciones_base,
    average="macro",
    zero_division=0
)

print(f"Accuracy modelo base: {accuracy_base:.2%}")
print(f"Macro F1 modelo base: {macro_f1_base:.4f}")

Accuracy modelo base: 100.00%
Macro F1 modelo base: 1.0000


In [14]:
print("=== ENTRENAMIENTO ===")
print(y_entrenamiento.value_counts().sort_index())

print("\n=== VALIDACIÓN ===")
print(y_validacion.value_counts().sort_index())

=== ENTRENAMIENTO ===
AgentAssignedTopic
Add New Line         6
Cancel Service       7
Change Plans         6
Data Usage           6
Deactivate           6
High Bill            7
Others               8
Poor Connectivity    6
Port Out             7
Roaming Plans        6
Name: count, dtype: int64

=== VALIDACIÓN ===
AgentAssignedTopic
Add New Line         2
Cancel Service       1
Change Plans         2
Data Usage           2
Deactivate           2
High Bill            1
Others               2
Poor Connectivity    2
Port Out             1
Roaming Plans        2
Name: count, dtype: int64


In [15]:
resultado_base = pd.DataFrame({
    "Conversación": X_validacion.values,
    "Real": y_validacion.values,
    "Predicción": predicciones_base
})

resultado_base["Correcto"] = (
    resultado_base["Real"] ==
    resultado_base["Predicción"]
)

resultado_base

,Conversación,Real,Predicción,Correcto
0,Low/poor connectivity during day-time. Custome...,Poor Connectivity,Poor Connectivity,True
1,Customer want to change location of SIM to Ban...,Roaming Plans,Roaming Plans,True
2,Customer called to ask about adding a new line...,Add New Line,Add New Line,True
3,Customer called in to enquire data usage polic...,Data Usage,Data Usage,True
4,Cus called in to change plan.,Change Plans,Change Plans,True
5,porting out because of high bill on Data Plans.,Port Out,Port Out,True
6,Customer wanted to deactivate existing plans.,Deactivate,Deactivate,True
7,Reported sad demise of customer.,Others,Others,True
8,Cus travelling for nect month. Checked roaming...,Roaming Plans,Roaming Plans,True
9,Deactivation requires customer authentication.,Deactivate,Deactivate,True


In [16]:
print(resultado_base["Correcto"].value_counts())

Correcto
True    17
Name: count, dtype: int64


revisar datos duplicados

In [17]:
# Crear DataFrames para comparar
df_entrenamiento = pd.DataFrame({
    "Conversación": X_entrenamiento.values,
    "Etiqueta": y_entrenamiento.values
})

df_validacion = pd.DataFrame({
    "Conversación": X_validacion.values,
    "Etiqueta": y_validacion.values
})

# Buscar conversaciones exactamente iguales
duplicados = df_validacion[
    df_validacion["Conversación"].isin(
        df_entrenamiento["Conversación"]
    )
]

print("Duplicados exactos entre entrenamiento y validación:")
print(len(duplicados))

duplicados

Duplicados exactos entre entrenamiento y validación:
1


,Conversación,Etiqueta
10,Checking IMEI status.,Others


In [18]:
X_cv_reentrenado = pd.concat(
    [
        X_train.reset_index(drop=True),
        X_corregido.reset_index(drop=True)
    ],
    ignore_index=True
)

y_cv_reentrenado = pd.concat(
    [
        y_train.reset_index(drop=True),
        y_corregido.reset_index(drop=True)
    ],
    ignore_index=True
)

print("Total datos base:", len(X_train))
print("Casos revisados agregados:", len(X_corregido))
print("Total datos reentrenamiento:", len(X_cv_reentrenado))

Total datos base: 82
Casos revisados agregados: 7
Total datos reentrenamiento: 89


In [19]:
pipeline_reentrenado_cv = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        )
    )
])

In [29]:
cv_reentrenamiento = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [30]:
scores_reentrenado_cv = cross_validate(
    pipeline_reentrenado_cv,
    X_cv_reentrenado,
    y_cv_reentrenado,
    cv=cv_reentrenamiento,
    scoring=[
        "accuracy",
        "f1_macro"
    ]
)

print(
    f"Accuracy promedio reentrenado: "
    f"{scores_reentrenado_cv['test_accuracy'].mean():.4f}"
)

print(
    f"Macro F1 promedio reentrenado: "
    f"{scores_reentrenado_cv['test_f1_macro'].mean():.4f}"
)

print(
    f"Std Accuracy: "
    f"{scores_reentrenado_cv['test_accuracy'].std():.4f}"
)

print(
    f"Std Macro F1: "
    f"{scores_reentrenado_cv['test_f1_macro'].std():.4f}"
)

Accuracy promedio reentrenado: 0.7399
Macro F1 promedio reentrenado: 0.7194
Std Accuracy: 0.0972
Std Macro F1: 0.0977


In [33]:
accuracy_base_cv = 0.8154
macro_f1_base_cv = 0.7740

accuracy_reentrenado_cv = scores_reentrenado_cv["test_accuracy"].mean()
macro_f1_reentrenado_cv = scores_reentrenado_cv["test_f1_macro"].mean()

comparacion_reentrenamiento = pd.DataFrame({
    "Modelo": [
        "Modelo base",
        "Modelo reentrenado"
    ],
    "Accuracy": [
        accuracy_base_cv,
        accuracy_reentrenado_cv
    ],
    "Macro F1": [
        macro_f1_base_cv,
        macro_f1_reentrenado_cv
    ]
})

comparacion_reentrenamiento["Diferencia Accuracy"] = (
    comparacion_reentrenamiento["Accuracy"]
    - accuracy_base_cv
)

comparacion_reentrenamiento["Diferencia Macro F1"] = (
    comparacion_reentrenamiento["Macro F1"]
    - macro_f1_base_cv
)

comparacion_reentrenamiento

,Modelo,Accuracy,Macro F1,Diferencia Accuracy,Diferencia Macro F1
0,Modelo base,0.815400,0.774000,0.000000,0.000000
1,Modelo reentrenado,0.739869,0.719429,-0.075531,-0.054571


In [34]:
if (
    accuracy_reentrenado_cv > accuracy_base_cv
    and macro_f1_reentrenado_cv > macro_f1_base_cv
):
    print("✅ El modelo reentrenado mejora al modelo base.")
    print("🚀 Se recomienda reemplazar el modelo base.")
else:
    print("⚠️ El modelo reentrenado no supera al modelo base.")
    print("✅ Se mantiene el modelo base en producción.")
    print("📊 Se recomienda continuar recopilando datos revisados.")

⚠️ El modelo reentrenado no supera al modelo base.
✅ Se mantiene el modelo base en producción.
📊 Se recomienda continuar recopilando datos revisados.
